In [1]:
from astroquery.mast import Mast, Observations
from astropy.io import fits
from astropy import table as tb
from astropy.time import Time

import numpy as np
from tqdm.notebook import tqdm

from glob import glob

directory = 'D:/jwst_data/prod'
dirstar = directory + '/mastDownload/JWST/**/'

In [2]:
obs    = Observations.query_criteria(
         obs_collection = 'JWST',
         instrument_name = 'MIRI/CORON',
         calib_level = '3',
         filters = '*;4QPM*'
)
filters = {}
for filt in set(obs['filters']):
    filters[filt] = obs[ obs['filters'] == filt]
objects = []
for obj in set(obs['target_name']):
    if np.all([obj in filters[filt]['target_name'] for filt in filters]):
        objects.append(obj)

In [3]:
for obj in tqdm(objects):
    obs = Observations.query_criteria(
          obs_collection = 'JWST',
          instrument_name = 'MIRI/CORON',
          calib_level = '3',
          target_name = obj,
          filters = '*;4QPM*'
    )
    if len(obs) == 0:
        print('nothing for ' + obj)
        continue
    t = [Observations.get_product_list(o) for o in obs]
    files = tb.unique(tb.vstack(t), keys = 'productFilename')
    print(f'-----{len(files)}------')
    manifest = Observations.download_products(
        files,
        curl_flag=False,
        calib_level = [3],
        download_dir = directory
    )

  0%|          | 0/7 [00:00<?, ?it/s]

-----460------
INFO: Found cached file D:/jwst_data/prod\mastDownload\JWST\jw01194-c1012_t001_miri_f1065c-mask1065\jw01194-c1012_20251223t195801_coron3_00001_asn.json with expected size 5533. [astroquery.query]
INFO: Found cached file D:/jwst_data/prod\mastDownload\JWST\jw01194-c1012_t001_miri_f1065c-mask1065\jw01194-c1012_t001_miri_f1065c-mask1065_i2d.fits with expected size 1990080. [astroquery.query]
INFO: Found cached file D:/jwst_data/prod\mastDownload\JWST\jw01194-c1012_t001_miri_f1065c-mask1065\jw01194-c1012_t001_miri_f1065c-mask1065_i2d.jpg with expected size 9823. [astroquery.query]
INFO: Found cached file D:/jwst_data/prod\mastDownload\JWST\jw01194-c1012_t001_miri_f1065c-mask1065\jw01194-c1012_t001_miri_f1065c-mask1065_psfstack.fits with expected size 14014080. [astroquery.query]
INFO: Found cached file D:/jwst_data/prod\mastDownload\JWST\jw01194-c1012_t001_miri_f1065c-mask1065\jw01194-c1012_t001_miri_f1065c-mask1065_psfstack.jpg with expected size 19730. [astroquery.query]
I

In [4]:
objects

[np.str_('HR8799'),
 np.str_('HR-2562'),
 np.str_('KAPPA-AND'),
 np.str_('51-ERI'),
 np.str_('BETA-PIC'),
 np.str_('GJ-758-revised'),
 np.str_('HD-141569A')]